# Inventory Intelligence — Kaggle training
CPU workflow. Attach the source ZIP as a private dataset. Enable Internet. Real M5 retail sales; stock and costs are simulated. See README and methodology before interpreting results.

In [ ]:
from pathlib import Path
import zipfile, shutil, subprocess, sys, os
WORK = Path('/kaggle/working/inventory-intelligence')
archives = list(Path('/kaggle/input').rglob('inventory-intelligence-source.zip'))
if not WORK.exists():
    if len(archives) != 1: raise RuntimeError('Attach exactly one project source ZIP dataset')
    with zipfile.ZipFile(archives[0]) as z:
        for member in z.infolist():
            dest = (Path('/kaggle/working') / member.filename).resolve()
            if not dest.is_relative_to(Path('/kaggle/working')): raise ValueError('Unsafe archive path')
        z.extractall('/kaggle/working')
os.chdir(WORK)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-e', '.[dev]'], check=True)

In [ ]:
DATA = WORK / 'data/raw'
# Alternatively set DATA to the folder containing official M5 evaluation sales and calendar.csv.
if not (DATA / 'sales_train_evaluation.csv').exists():
    subprocess.run([sys.executable, 'scripts/download_m5.py', '--output', str(DATA), '--commit', '72b8e7fd3b565b3c538adcb1d1a05117d8562d7e'], check=True)
subprocess.run([sys.executable, '-m', 'pytest', '-q'], check=True)

In [ ]:
from datetime import datetime, timezone
RUN = WORK / 'runs' / datetime.now(timezone.utc).strftime('kaggle-%Y%m%dT%H%M%SZ')
subprocess.run([sys.executable, '-m', 'inventory_intelligence.cli', '--data', str(DATA), '--config', 'configs/kaggle.json', '--output', str(RUN)], check=True)

In [ ]:
import pandas as pd
from IPython.display import display
display(pd.read_csv(RUN / 'policy_metrics.csv'))
display(pd.read_csv(RUN / 'forecast_metrics.csv'))
print((RUN / 'SUMMARY.md').read_text())
shutil.make_archive('/kaggle/working/inventory-results', 'zip', RUN)
print('Download inventory-results.zip from notebook output; open report.html locally.')

## Interpretation
Compare service and cost jointly. Check interval coverage, shortage-cost sensitivity, and paired comparisons. The final test is for evaluation, not repeated model tuning. A larger training configuration is supplied, not a guarantee of better results.